# ReTone — Polyphonic Instrument Conversion: Training Pipeline

**Goal:** take a recording of *any* instrument playing *anything* — a piano playing chords, a
strummed guitar, a string section — and re-render it as a **different instrument**, keeping the
performance identical. Same notes, same timing, same dynamics. Only the instrument changes.

The mental model is an electric keyboard's **tone switch**. You play a phrase, then flip from
PIANO to STRINGS to ORGAN. The performance never changes; only the sound engine does. We want that,
but the "performance" arrives as *audio* rather than as key presses.

---

## Where this fits in ReTone

| Engine | Handles | Status |
|---|---|---|
| **1 — DDSP mono** | monophonic pitched audio (vocals, solo violin/flute/sax/trumpet) | ✅ shipped, sounds good |
| **2 — this notebook** | **polyphonic pitched audio (piano, guitar, strings)** | 🔨 what we're building |
| **3 — Inverse Drum Machine** | drums (unpitched percussion) | planned |

Engine 1 works because DDSP has a physically-correct inductive bias: *one f0 → one harmonic series*.
That is exactly why it sounds convincing, and exactly why it cannot do chords. Polyphony breaks the
assumption at its root, so Engine 2 needs a different architecture — not a bigger DDSP.

---

## What we already tried, and what it taught us

**AFTER** (ACIDS-IRCAM latent diffusion, [arXiv 2408.00196](https://arxiv.org/abs/2408.00196)) —
polyphonic-capable audio→audio timbre transfer. We wired it end-to-end and listened. Verdict:
clearly worse than the monophonic DDSP.

Two lessons, both of which shape this notebook:

1. **Instrument identity belongs in the weights.** AFTER picks its target timbre at *inference*
   from a reference audio clip, so one general model has to be able to sound like everything. DDSP's
   violin decoder can *only* produce violin — that constraint is where the quality comes from.
   → **This notebook trains one model per target instrument.**

2. **A general audio→audio morph is a harder problem than it needs to be.** It has to
   simultaneously figure out *what was played* and *how the target sounds*, from raw waveform, with
   no explicit representation of either. Splitting those two jobs makes each one tractable.

---

## The architecture

```
   polyphonic audio in                                        target instrument audio out
          │                                                              ▲
          │                                                              │
   ┌──────▼─────────────────┐    ┌───────────────┐    ┌──────────────────┴──────┐
   │  STAGE 1               │    │  VOICING      │    │  STAGE 2                │
   │  Transcription         │───▶│  POLICY       │───▶│  Neural renderer        │
   │  audio → note events   │    │  (musical)    │    │  notes → target audio   │
   │                        │    │               │    │                         │
   │  PRETRAINED            │    │  RULES        │    │  ◀── WE TRAIN THIS      │
   │  (Basic Pitch / etc.)  │    │               │    │      one per instrument │
   └────────────────────────┘    └───────────────┘    └─────────────────────────┘
```

**Stage 1 is not trained by us** — polyphonic transcription is a solved-enough problem with good
pretrained models. **Stage 2 is the only thing this notebook trains**, and it is a genuinely
tractable supervised learning problem, because we can manufacture unlimited perfectly-aligned
training data (§7).

### Why this decomposition is the right call

- **Performance preservation is explicit, not hoped-for.** The note events *are* the performance.
  Nothing can drift, because timing and pitch are handed to the renderer as data.
- **Training data is unlimited.** Any MIDI file rendered through any sample library gives a
  perfectly-aligned `(notes, audio)` pair. No alignment error, no annotation cost. This is the
  single biggest practical advantage.
- **Per-instrument identity, like DDSP.** Each target instrument gets its own trained renderer.
- **The output is editable.** Note events are a first-class object in the DAW — the user can fix a
  wrong note *before* rendering, which a black-box audio→audio model can never offer.

### The honest cost

Transcription errors propagate. If Stage 1 misses a note, Stage 2 faithfully renders the miss.
Everything not captured in the note representation (room tone, bow noise, breath, exact
articulation) is discarded and must be *re-invented* by the renderer. §5 is about measuring how
much this actually costs on your material — do that before training anything.

---

# Part I · What a "performance" actually is

Before writing code it is worth being precise about what we are extracting and re-rendering,
because the representation choice determines the quality ceiling of the whole system.

## The note event

Western music notation, MIDI, and every DAW piano roll all encode roughly the same object:

| Field | Meaning | Range | Why it matters here |
|---|---|---|---|
| **pitch** | which note | MIDI 0–127, 60 = middle C | Semitone = one integer step. Equal temperament: `f = 440 · 2^((p−69)/12)` |
| **onset** | when it starts | seconds | Attack timing carries *groove*. Errors here are very audible |
| **offset** | when it stops | seconds | Determines sustain/decay. Piano offsets are ambiguous (decay is gradual) |
| **velocity** | how hard struck | 1–127 | **Not just loudness** — a hard piano strike is brighter, not merely louder |
| **pitch bend** | continuous deviation | cents | Essential for guitar bends, vibrato, fretless/vocal glides |
| **pedal** | sustain state | on/off (or 0–127) | Piano-specific and *large*: pedal changes which strings resonate sympathetically |

**Velocity is the subtle one.** It is tempting to treat it as a gain multiplier, but on real
instruments dynamics change *timbre*, not just level: a fortissimo piano note has far more upper
harmonic energy than a pianissimo one at the same pitch. A renderer that treats velocity as volume
sounds flat and synthetic. Our model gets velocity as an explicit conditioning channel so it can
learn the timbral consequence.

## Polyphony, and why it is genuinely hard

**Monophonic**: at most one note sounding at a time. One f0, one harmonic series. Pitch tracking is
well-posed; DDSP's inductive bias is exactly correct.

**Polyphonic**: multiple simultaneous notes. Now the spectrum is a *sum* of harmonic series, and:

- **Harmonics collide.** A perfect fifth (C3 + G3) has G3's fundamental landing exactly on C3's 3rd
  harmonic. An octave is worse — every one of the upper note's harmonics coincides with one of the
  lower's. There is genuinely no way to attribute that shared energy to one note or the other from
  the spectrum alone.
- **Octave errors are the classic failure.** The above is why transcribers systematically confuse
  a note with the note an octave above/below it.
- **Note count is unknown.** Was that a 3-note or 4-note chord? The model has to infer it.

This is precisely why we do not ask a single network to do timbre transfer directly on polyphonic
audio: it would have to solve this implicitly *and* re-synthesize, with no supervision on the hard
part.

## The instrumentation problem (the part that is musical, not technical)

Here is a question no neural network answers for you:

> A pianist plays a 6-note chord spanning three octaves. You asked for **violin**.
> A violin has four strings and can sustain at most two notes comfortably.
> **What should come out?**

The honest answer is *it depends on what you meant*, and there are several legitimate choices:

| Policy | Result | When it's right |
|---|---|---|
| **Ensemble** | Treat "violin" as a violin *section*; render all 6 notes | Cinematic, lush. Usually the best default |
| **Reduce voicing** | Keep root + melody, drop inner voices | Idiomatic solo violin. Musical, but discards material |
| **Arpeggiate** | Spread the chord over time | Idiomatic for harp/guitar, a stylistic choice for violin |
| **Split to sections** | Vln I / Vln II / Viola / Cello by register | The real orchestration answer. Most work |

This is **arrangement**, a musical decision that predates ML by centuries. Our pipeline makes it an
explicit, inspectable step (§6) rather than something a black-box model does unpredictably. Default:
**ensemble**, because it preserves all the musical content and is what users expect from
"make this sound like strings."

### Instrument ranges (used by the voicing policy)

Rendering notes outside an instrument's physical range is a common and very audible mistake.

| Instrument | Low | High | MIDI | Notes |
|---|---|---|---|---|
| Piano | A0 | C8 | 21–108 | The reference wide range |
| Violin | G3 | ~A7 | 55–105 | 4 strings: G3 D4 A4 E5 |
| Viola | C3 | ~E6 | 48–88 | A fifth below violin |
| Cello | C2 | ~C6 | 36–84 | |
| Double bass | E1 | ~G4 | 28–67 | Sounds an octave below written |
| Guitar (ac.) | E2 | ~E6 | 40–88 | Sounds an octave below written |
| Bass guitar | E1 | ~G4 | 28–67 | 4-string standard |
| Flute | C4 | ~D7 | 60–98 | |
| Trumpet (Bb) | E3 | ~C6 | 52–84 | |
| Rhodes / E.piano | ~E1 | ~E7 | 28–100 | |

---

# Part II · Stage 1 — Transcription (pretrained; we do not train this)

Turning polyphonic audio into note events. Use a pretrained model; this is a well-studied task and
training your own is a whole research project with no upside for us.

## Choosing a transcriber

| Model | Scope | Framework | License | Use it when |
|---|---|---|---|---|
| **[Basic Pitch](https://github.com/spotify/basic-pitch)** (Spotify) | instrument-agnostic polyphonic, **outputs pitch bends** | TF / CoreML / ONNX | Apache-2.0 | **Default.** Unknown or non-piano instruments. Tiny (~20 MB), fast, CPU-fine |
| **Piano transcribers** (Onsets & Frames lineage, MAESTRO-trained) | piano only | torch | varies | **Piano input.** Much more accurate than general models, and gives *velocity + pedal* |
| **[MT3](https://github.com/magenta/mt3)** (Magenta) | multi-instrument, transformer | JAX/T5 | Apache-2.0 | Multi-instrument mixtures. Heavy; awkward to run |

**Recommended policy:** piano-specific model for piano stems, Basic Pitch for everything else. We
already run Basic Pitch elsewhere in ReTone (`backend/app/services/analysis.py`), so the dependency
is familiar.

**Note on our input:** ReTone transcribes *separated stems*, not full mixes. That is Basic Pitch's
documented sweet spot and materially easier than polyphonic multi-instrument transcription.

## §5 — Measure transcription quality FIRST

Transcription error is the hard ceiling on the whole pipeline. Measure it on *your* material before
training anything downstream. The check below transcribes, re-renders the notes with a plain
sampler, and lets you listen — if the re-render already sounds wrong, no amount of Stage-2 training
will fix it.

In [ ]:
# Stage 1 sanity check: how good is transcription on YOUR audio?
# Run this BEFORE training anything. It is the cheapest possible way to find out
# whether the two-stage approach can work on your material.
#
# !pip install basic-pitch pretty_midi librosa soundfile

import numpy as np, pretty_midi, librosa

AUDIO_IN = "path/to/your/polyphonic_stem.wav"   # ← a piano or guitar stem

def transcribe(audio_path, out_midi="transcribed.mid"):
    """Polyphonic audio → note events, via Basic Pitch."""
    from basic_pitch.inference import predict
    from basic_pitch import ICASSP_2022_MODEL_PATH

    model_out, midi_data, note_events = predict(
        audio_path,
        ICASSP_2022_MODEL_PATH,
        onset_threshold=0.5,      # ↑ = fewer spurious notes, ↓ = fewer missed notes
        frame_threshold=0.3,
        minimum_note_length=58,   # ms — filters transcription chatter
        # melodia_trick=True,     # helps on monophonic-ish input
    )
    midi_data.write(out_midi)
    return midi_data, note_events

def describe(midi_data):
    """Report the things that predict downstream quality."""
    notes = [n for inst in midi_data.instruments for n in inst.notes]
    if not notes:
        print("⚠️  NO NOTES TRANSCRIBED — check input level / thresholds"); return
    pitches = np.array([n.pitch for n in notes])
    vels    = np.array([n.velocity for n in notes])
    durs    = np.array([n.end - n.start for n in notes])

    # Max simultaneous notes = the polyphony the renderer must handle
    events = sorted([(n.start, 1) for n in notes] + [(n.end, -1) for n in notes])
    cur = peak = 0
    for _, d in events:
        cur += d; peak = max(peak, cur)

    print(f"  notes            : {len(notes)}")
    print(f"  pitch range      : {pitches.min()}–{pitches.max()} "
          f"({pretty_midi.note_number_to_name(int(pitches.min()))}–"
          f"{pretty_midi.note_number_to_name(int(pitches.max()))})")
    print(f"  max polyphony    : {peak} simultaneous notes")
    print(f"  median duration  : {np.median(durs)*1000:.0f} ms")
    print(f"  velocity spread  : {vels.min()}–{vels.max()} "
          f"{'  ⚠️ flat — Basic Pitch velocity is weak' if vels.std() < 5 else ''}")
    print(f"  very short notes : {(durs < 0.05).sum()}  (likely spurious)")

# midi_data, _ = transcribe(AUDIO_IN)
# describe(midi_data)
print("Fill in AUDIO_IN and uncomment to run.")


In [ ]:
# The listening test that actually matters: re-render the transcription with a plain
# sampler and compare to the original. This isolates Stage-1 error from Stage-2 quality.
#
# If this sounds like the right performance (even with a cheap piano sound), the pipeline
# is viable. If notes are missing/wrong here, fix transcription before going further.

import subprocess, shutil

def render_midi_reference(midi_path, out_wav, soundfont="GeneralUser-GS.sf2", sr=44100):
    """MIDI → audio via FluidSynth. A neutral baseline, not the trained renderer."""
    if shutil.which("fluidsynth") is None:
        print("install fluidsynth:  apt-get install -y fluidsynth   /   brew install fluid-synth")
        return None
    subprocess.run(["fluidsynth", "-ni", "-F", out_wav, "-r", str(sr), soundfont, midi_path],
                   check=True, capture_output=True)
    return out_wav

# render_midi_reference("transcribed.mid", "transcribed_reference.wav")
# → listen to transcribed_reference.wav against your original.
#   Judge ONLY notes/timing/dynamics here; the timbre is meant to be plain.
print("Uncomment after transcribing.")


---

# Part III · The voicing policy (§6)

The explicit musical layer from Part I. Given transcribed notes and a target instrument, decide what
the target should actually play — clamped to the instrument's real range, with a polyphony policy.

This is deliberately **rules-based and inspectable**. It is arrangement, not perception; a neural
network would make it unpredictable for no benefit.

In [ ]:
"""Voicing policy: map transcribed notes onto what the target instrument can play."""
import copy
import numpy as np
import pretty_midi

# (low, high) MIDI range per instrument — see the table in Part I
INSTRUMENT_RANGES = {
    "piano":        (21, 108),
    "violin":       (55, 105),
    "viola":        (48, 88),
    "cello":        (36, 84),
    "double_bass":  (28, 67),
    "strings":      (28, 105),   # full section: bass → violin
    "acoustic_guitar": (40, 88),
    "electric_guitar": (40, 88),
    "bass_guitar":  (28, 67),
    "flute":        (60, 98),
    "trumpet":      (52, 84),
    "rhodes":       (28, 100),
    "organ":        (24, 108),
}

def apply_voicing(midi_data, target, policy="ensemble", max_voices=None):
    """
    policy:
      "ensemble" — keep every note (target treated as a section). DEFAULT.
      "reduce"   — keep the `max_voices` most important notes per chord
                   (lowest = harmonic root, highest = melody — both carry the most information)
      "octave"   — fold out-of-range notes by octaves instead of clamping
    """
    lo, hi = INSTRUMENT_RANGES[target]
    out = copy.deepcopy(midi_data)

    for inst in out.instruments:
        kept = []
        for n in inst.notes:
            p = n.pitch
            if policy == "octave":
                # Preserve pitch CLASS by shifting whole octaves — musically far better
                # than clamping, which collapses distinct notes onto one pitch.
                while p < lo: p += 12
                while p > hi: p -= 12
                if lo <= p <= hi:
                    n.pitch = p; kept.append(n)
            else:
                if lo <= p <= hi:
                    kept.append(n)
                # else: silently dropped — out of the instrument's physical range
        inst.notes = kept

    if policy == "reduce" and max_voices:
        for inst in out.instruments:
            inst.notes = _reduce_polyphony(inst.notes, max_voices)
    return out

def _reduce_polyphony(notes, max_voices, tol=0.05):
    """Thin simultaneous notes to `max_voices`, keeping the outer voices.

    Musical rationale: in a chord the BASS defines the harmony and the TOP defines the
    melody. Inner voices are the most expendable — this is standard reduction practice.
    """
    notes = sorted(notes, key=lambda n: n.start)
    groups, cur = [], []
    for n in notes:
        if cur and abs(n.start - cur[0].start) > tol:
            groups.append(cur); cur = []
        cur.append(n)
    if cur: groups.append(cur)

    out = []
    for g in groups:
        if len(g) <= max_voices:
            out.extend(g); continue
        g = sorted(g, key=lambda n: n.pitch)
        picked = [g[0], g[-1]]                       # bass + melody first
        for n in reversed(g[1:-1]):                  # then from the top down
            if len(picked) >= max_voices: break
            picked.append(n)
        out.extend(picked)
    return sorted(out, key=lambda n: n.start)

print("Voicing policy ready. Targets:", ", ".join(INSTRUMENT_RANGES))


---

# Part IV · Stage 2 — the renderer we train

## The architecture, and why

```
   note events
        │
        ▼
  ┌─────────────┐   piano roll: (3, 128 pitches, T frames)
  │ PIANO ROLL  │   channels = [onset, sustain, velocity]
  └─────┬───────┘
        │
        ▼
  ┌─────────────┐   ◀── THE TRAINED MODEL (one per instrument, ~15M params)
  │  RENDERER   │       conv stack + transformer over time
  └─────┬───────┘
        │
        ▼
   mel spectrogram: (128 mels, T frames)
        │
        ▼
  ┌─────────────┐   ◀── PRETRAINED, FROZEN (BigVGAN-v2, MIT license)
  │   VOCODER   │
  └─────┬───────┘
        ▼
     waveform @ 44.1 kHz
```

**Why mel-spectrogram as the intermediate,** rather than predicting the waveform directly:

- Raw waveform prediction means learning phase, which is hard, slow, and mostly perceptually
  irrelevant. Mel discards phase and keeps what the ear cares about.
- Neural vocoders solve mel→waveform *very* well and are available pretrained. We reuse that work
  instead of repeating it — the vocoder stays frozen and never trains.
- The whole problem becomes a **frame-aligned sequence-to-sequence regression**: 128 conditioning
  values per frame in, 128 mel bins per frame out, same frame rate. That is a simple, stable,
  fast-training setup — no attention alignment to learn, no length mismatch.

**The frame-rate trick:** we set the piano-roll frame rate equal to the vocoder's mel hop
(44100/512 ≈ 86.13 fps). Input and output are then the same length, frame for frame.

### Alternatives considered

| Approach | Why not primary |
|---|---|
| Per-note DDSP stacking (reuse Engine 1, render each note, sum) | Cheapest — no training. But summed independent voices lack sympathetic resonance and shared room; attacks smear. **Worth trying as a free baseline** (§14) |
| Direct audio→audio diffusion (AFTER, WaveTransfer) | Already tested; worse, and identity is not in the weights |
| Autoregressive token LM over codec tokens (MusicGen-style) | Excellent quality but a *generator*, not a renderer — hard to force exact performance preservation. Also large and slow |
| Diffusion on mel | Higher ceiling than regression, ~10× slower to train and to sample. Consider as v2 once the regression baseline works |

Starting with deterministic regression is deliberate: it trains in hours not days, has no sampling
loop, and gives a working end-to-end system to measure against.

## §7 — Training data: the decisive advantage

We need aligned `(note events, target instrument audio)` pairs. Unlike almost every other audio-ML
task, **we can manufacture these perfectly and without limit**:

> Take any MIDI file → render it through a sample library of the target instrument →
> you now have audio that is *exactly*, sample-accurately aligned to those notes.

No annotation, no alignment error, no labelling cost. This is why Stage 2 is tractable.

### Two sources, used together

**A. Real recordings with aligned MIDI** — highest realism, limited quantity:

| Dataset | Instrument | Hours | License | Link |
|---|---|---|---|---|
| **MAESTRO v3** | piano (Disklavier, ~3 ms alignment) | 199 | **CC BY-NC-SA** ⚠️ non-commercial | [magenta](https://magenta.withgoogle.com/datasets/maestro) |
| **Slakh2100** | 34 instrument classes, per-stem + MIDI | 145 | CC BY 4.0 ✅ | [zenodo](https://zenodo.org/records/4599666) |
| **GuitarSet** | acoustic guitar | 3 | CC BY 4.0 ✅ | [zenodo](https://zenodo.org/records/3371780) |
| **URMP** | 13 orchestral instruments | 1.3 | research ✅ | [Rochester](https://labsites.rochester.edu/air/datasets/URMP.html) |
| **CocoChorales** | 13 instruments, synthetic | 1400 | CC BY 4.0 ✅ | [magenta](https://magenta.tensorflow.org/datasets/cocochorales) |
| **E-GMD** | drums | 444 | CC BY 4.0 ✅ | [magenta](https://magenta.withgoogle.com/datasets/e-gmd) |

⚠️ **MAESTRO is CC BY-NC-SA** — fine for research/personal, blocks commercial use. Slakh's piano
stems are the commercially-safe alternative.

**B. Self-rendered from MIDI** — unlimited quantity, any instrument:

- MIDI source: [Lakh MIDI Dataset](https://colinraffel.com/projects/lmd/) (176k files)
- Renderers: `fluidsynth --fast-render` (SF2) or `sfizz_render` (SFZ, better quality)
- Libraries (all permissive): [Salamander Grand Piano](https://freepats.zenvoid.org/Piano/SalamanderGrandPiano/)
  (CC BY 3.0), [Virtual Playing Orchestra](http://virtualplayingorchestra.com/) (CC BY 3.0),
  [GeneralUser GS](https://www.schristiancollins.com/generaluser.php) (permissive, commercial OK)

**The sim-to-real gap is real.** A model trained only on sample-library renders learns that
library's character, including its lack of room and mic variation. Two mitigations, both proven in
the literature:
1. **Preset/augmentation diversity** — randomize reverb IR, EQ, mic distance per render. Zehren et al.
   ([arXiv 2407.19823](https://arxiv.org/html/2407.19823)) found diversity matters *more than raw
   hours* past a few hundred hours.
2. **Fine-tune on real recordings last** — pretrain on synthetic, then a short pass on real audio.

Recommended per instrument: **~20–50 h synthetic** (varied) **+ whatever real data exists**, then
fine-tune on the real portion.

---

# Part V · Implementation

## §8 — Configuration

Every downstream cell reads `CFG`. To train a different instrument, change `INSTRUMENT` and re-run
from §9.

**The mel parameters are not free choices.** They must match the pretrained vocoder's expectations
exactly — a mismatch produces garbage output with no error message. These match
[`nvidia/bigvgan_v2_44khz_128band_512x`](https://huggingface.co/nvidia/bigvgan_v2_44khz_128band_512x).

In [ ]:
import os, sys, json, math, pathlib, subprocess, random

INSTRUMENT = "strings"       # ← the TARGET instrument this model renders
                             #   strings | piano | acoustic_guitar | electric_guitar
                             #   bass_guitar | rhodes | organ | violin | cello ...

ROOT      = pathlib.Path("/workspace/retone_poly").resolve()
REPO_DIR  = pathlib.Path("..").resolve()          # the retone repo (notebook lives in retone/notebooks)
DATA_DIR  = ROOT / "data" / INSTRUMENT
CACHE_DIR = ROOT / "cache" / INSTRUMENT           # preprocessed (piano_roll, mel) pairs
CKPT_DIR  = ROOT / "ckpt" / INSTRUMENT
OUT_DIR   = ROOT / "artifacts" / INSTRUMENT
for d in (DATA_DIR/"midi", DATA_DIR/"audio", CACHE_DIR, CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    # ── audio / mel — MUST match the pretrained vocoder ───────────────────
    sample_rate = 44100,
    n_fft       = 2048,
    hop_length  = 512,      # → frame rate 44100/512 = 86.13 fps
    win_length  = 2048,
    n_mels      = 128,
    fmin        = 0,
    fmax        = 22050,

    # ── conditioning ─────────────────────────────────────────────────────
    n_pitches   = 128,      # full MIDI range
    n_cond_ch   = 3,        # [onset, sustain, velocity]

    # ── model ────────────────────────────────────────────────────────────
    hidden      = 512,
    n_layers    = 8,
    n_heads     = 8,
    dropout     = 0.1,

    # ── training ─────────────────────────────────────────────────────────
    seq_frames  = 512,      # ~6 s per training example at 86 fps
    batch_size  = 16,       # fits 24 GB comfortably; raise on bigger cards
    lr          = 3e-4,
    max_steps   = 200_000,
    warmup      = 2_000,
    val_every   = 2_000,
    save_every  = 5_000,
    num_workers = 8,
    seed        = 940513,
)
CFG["frame_rate"] = CFG["sample_rate"] / CFG["hop_length"]

random.seed(CFG["seed"])
print(f"target instrument : {INSTRUMENT}")
print(f"frame rate        : {CFG['frame_rate']:.2f} fps")
print(f"training example  : {CFG['seq_frames']} frames = {CFG['seq_frames']/CFG['frame_rate']:.1f} s")
print(f"root              : {ROOT}")


## §9 — Environment

PyTorch + audio tooling + the vocoder. BigVGAN is fetched from HuggingFace and stays **frozen** throughout — we never train it.

In [ ]:
# GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU — training will be impractically slow"

# Core stack
!pip -q install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install pretty_midi librosa soundfile numpy scipy tqdm tensorboard einops

# Stage-1 transcription (inference only)
!pip -q install basic-pitch

# Vocoder deps (BigVGAN)
!pip -q install "huggingface_hub>=0.23"

# MIDI rendering for synthetic data
!which fluidsynth || (apt-get -qq update && apt-get -qq install -y fluidsynth)
!which ffmpeg     || (apt-get -qq update && apt-get -qq install -y ffmpeg)

import torch
print(f"\ntorch {torch.__version__} | cuda {torch.cuda.is_available()}",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
# BigVGAN — pretrained neural vocoder (mel → waveform). NVIDIA, MIT license.
# Frozen: it is a fixed, high-quality decoder. We only train the piano-roll → mel model.
#
# Model card: https://huggingface.co/nvidia/bigvgan_v2_44khz_128band_512x
#   44.1 kHz · 128 mel bands · hop 512  ← exactly our CFG, by design

BIGVGAN_REPO = "nvidia/bigvgan_v2_44khz_128band_512x"

!git clone -q https://github.com/NVIDIA/BigVGAN.git {str(ROOT/"BigVGAN")} 2>/dev/null || echo "already cloned"
sys.path.insert(0, str(ROOT/"BigVGAN"))

def load_vocoder(device="cuda"):
    """Load BigVGAN, frozen and in eval mode."""
    import bigvgan
    v = bigvgan.BigVGAN.from_pretrained(BIGVGAN_REPO, use_cuda_kernel=False)
    v.remove_weight_norm()
    v = v.eval().to(device)
    for p in v.parameters():
        p.requires_grad = False
    return v

# Verify the vocoder's mel config matches CFG — a silent mismatch here means
# the trained model's output will be decoded as noise.
try:
    _v = load_vocoder("cpu")
    h = _v.h
    print("vocoder config:")
    for k in ("sampling_rate", "num_mels", "n_fft", "hop_size", "win_size", "fmin", "fmax"):
        print(f"  {k:14} = {h.get(k) if hasattr(h,'get') else getattr(h,k,'?')}")
    print("\n⚠️  These MUST equal CFG's sample_rate/n_mels/n_fft/hop_length/win_length/fmin/fmax")
    del _v
except Exception as e:
    print("vocoder load failed:", e)


## §10 — Mel spectrogram

Must match BigVGAN's training transform exactly. BigVGAN ships the canonical implementation
(`meldataset.mel_spectrogram`) — use it rather than rolling our own, so there is no chance of a
subtle mismatch in windowing, padding, or log scaling.

In [ ]:
import torch, numpy as np

from meldataset import mel_spectrogram as _bigvgan_mel   # from the cloned BigVGAN repo

def audio_to_mel(wav: torch.Tensor, cfg=CFG) -> torch.Tensor:
    """(1, N) waveform in [-1,1]  →  (n_mels, T) log-mel, BigVGAN-compatible."""
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    mel = _bigvgan_mel(
        wav,
        n_fft=cfg["n_fft"], num_mels=cfg["n_mels"],
        sampling_rate=cfg["sample_rate"], hop_size=cfg["hop_length"],
        win_size=cfg["win_length"], fmin=cfg["fmin"], fmax=cfg["fmax"],
        center=False,
    )
    return mel.squeeze(0)

@torch.no_grad()
def mel_to_audio(mel: torch.Tensor, vocoder) -> torch.Tensor:
    """(n_mels, T) log-mel → (N,) waveform."""
    if mel.dim() == 2:
        mel = mel.unsqueeze(0)
    return vocoder(mel.to(next(vocoder.parameters()).device)).squeeze().cpu()

# Round-trip test: audio → mel → audio. This is the QUALITY CEILING of the whole
# system — the trained model can never beat the vocoder's own reconstruction.
def test_vocoder_roundtrip(wav_path, vocoder):
    import soundfile as sf, librosa
    y, _ = librosa.load(wav_path, sr=CFG["sample_rate"], mono=True)
    y = torch.from_numpy(y).float()
    mel   = audio_to_mel(y)
    recon = mel_to_audio(mel, vocoder)
    n = min(len(y), len(recon))
    print(f"  in {len(y)/CFG['sample_rate']:.1f}s → mel {tuple(mel.shape)} → out {len(recon)/CFG['sample_rate']:.1f}s")
    sf.write("vocoder_roundtrip.wav", recon[:n].numpy(), CFG["sample_rate"])
    print("  wrote vocoder_roundtrip.wav — listen: this is the best your model can sound")

print("Mel transform ready.")


## §11 — Piano roll

The conditioning representation. Three channels per pitch per frame:

| Channel | Encodes | Why |
|---|---|---|
| **onset** | a short pulse at note start | Attacks are perceptually critical and need to be *sharp* — a sustain-only encoding gives the model no crisp attack cue |
| **sustain** | 1 while the note is held | Which notes are sounding |
| **velocity** | velocity/127, held for the note | Lets the model learn the *timbral* consequence of dynamics, not just level |

Onset is given a short decaying ramp rather than a single frame: at 86 fps one frame is 11.6 ms, and
a single-frame spike is a very sparse gradient signal. A few-frame ramp trains much more stably.

In [ ]:
import numpy as np, pretty_midi

def midi_to_pianoroll(midi_data, n_frames=None, cfg=CFG, onset_frames=3):
    """pretty_midi object → (3, 128, T) float32 conditioning tensor."""
    fr = cfg["frame_rate"]
    notes = [n for inst in midi_data.instruments if not inst.is_drum for n in inst.notes]
    if n_frames is None:
        end = max((n.end for n in notes), default=0.0)
        n_frames = int(math.ceil(end * fr)) + 1

    roll = np.zeros((3, cfg["n_pitches"], n_frames), dtype=np.float32)
    for n in notes:
        if not (0 <= n.pitch < cfg["n_pitches"]):
            continue
        s = int(round(n.start * fr))
        e = int(round(n.end   * fr))
        s = max(0, min(s, n_frames - 1))
        e = max(s + 1, min(e, n_frames))

        roll[1, n.pitch, s:e] = 1.0                 # sustain
        roll[2, n.pitch, s:e] = n.velocity / 127.0  # velocity

        # Onset: decaying ramp over `onset_frames`, not a single spike.
        for k in range(min(onset_frames, e - s)):
            roll[0, n.pitch, s + k] = max(roll[0, n.pitch, s + k], 1.0 - k / onset_frames)
    return roll

def pianoroll_summary(roll):
    poly = roll[1].sum(axis=0)
    active = poly[poly > 0]
    print(f"  shape {roll.shape}  ({roll.shape[2]/CFG['frame_rate']:.1f}s)")
    print(f"  onsets {int((roll[0] > 0.99).sum())}   "
          f"mean polyphony {active.mean():.1f}   max {int(poly.max())}")

print("Piano-roll encoder ready.")


## §12 — Building the dataset

Two paths, and you should use both:

- **`build_synthetic()`** — render MIDI through a sample library. Unlimited, any instrument,
  perfectly aligned. Augment aggressively (§7) so the model does not overfit one library's character.
- **`build_from_real()`** — real recordings that ship with aligned MIDI (MAESTRO, Slakh, GuitarSet).
  Fewer hours, far more realistic. Use for the final fine-tune.

Both produce the same cached artifact: `(piano_roll, mel)` pairs as `.npz`.

In [ ]:
import numpy as np, soundfile as sf, librosa, pretty_midi, torch
from tqdm.auto import tqdm

SOUNDFONTS = {
    # instrument → (soundfont path, GM program number)
    "piano":            ("soundfonts/SalamanderGrandPiano.sf2", 0),
    "rhodes":           ("soundfonts/GeneralUser-GS.sf2", 4),
    "organ":            ("soundfonts/GeneralUser-GS.sf2", 19),
    "acoustic_guitar":  ("soundfonts/GeneralUser-GS.sf2", 24),
    "electric_guitar":  ("soundfonts/GeneralUser-GS.sf2", 27),
    "bass_guitar":      ("soundfonts/GeneralUser-GS.sf2", 33),
    "violin":           ("soundfonts/VirtualPlayingOrchestra.sf2", 40),
    "cello":            ("soundfonts/VirtualPlayingOrchestra.sf2", 42),
    "strings":          ("soundfonts/VirtualPlayingOrchestra.sf2", 48),
}

def render_midi(midi_path, out_wav, soundfont, program=None, sr=CFG["sample_rate"]):
    """MIDI → audio via FluidSynth offline render."""
    if program is not None:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
        for inst in pm.instruments:
            if not inst.is_drum:
                inst.program = program
        tmp = str(out_wav) + ".mid"
        pm.write(tmp); midi_path = tmp
    subprocess.run(
        ["fluidsynth", "-ni", "-F", str(out_wav), "-r", str(sr), "-g", "0.7",
         str(soundfont), str(midi_path)],
        check=True, capture_output=True)
    return out_wav

def augment(y, sr):
    """Randomized room/EQ/level so the model does not memorize one library's character.
    Diversity here matters more than raw hours (Zehren et al., arXiv 2407.19823)."""
    import scipy.signal as sps
    # synthetic exponential-decay reverb
    if random.random() < 0.7:
        rt60 = random.uniform(0.15, 1.2)
        L = int(rt60 * sr)
        ir = np.random.randn(L) * np.exp(-np.linspace(0, 6, L))
        ir[0] = 1.0
        y = sps.fftconvolve(y, ir / np.abs(ir).sum(), mode="same")
    # gentle random tilt EQ
    if random.random() < 0.5:
        b, a = sps.butter(2, random.uniform(0.05, 0.45), btype=random.choice(["low", "high"]))
        y = 0.7 * y + 0.3 * sps.lfilter(b, a, y)
    # level
    y = y * random.uniform(0.5, 1.0)
    peak = np.abs(y).max() + 1e-9
    return (y / peak * random.uniform(0.5, 0.95)).astype(np.float32)

def build_pairs(midi_paths, audio_paths, out_dir, do_augment=False, limit=None):
    """Cache aligned (piano_roll, mel) pairs to .npz."""
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for mp, ap in tqdm(list(zip(midi_paths, audio_paths))[:limit], desc="pairs"):
        try:
            y, _ = librosa.load(str(ap), sr=CFG["sample_rate"], mono=True)
            if do_augment: y = augment(y, CFG["sample_rate"])
            mel = audio_to_mel(torch.from_numpy(y).float()).numpy()
            roll = midi_to_pianoroll(pretty_midi.PrettyMIDI(str(mp)), n_frames=mel.shape[1])
            T = min(roll.shape[2], mel.shape[1])
            if T < CFG["seq_frames"]:      # too short to train on
                continue
            np.savez_compressed(out_dir / f"pair_{n:06d}.npz",
                                roll=roll[:, :, :T].astype(np.float16),
                                mel=mel[:, :T].astype(np.float16))
            n += 1
        except Exception as e:
            print(f"  skip {pathlib.Path(mp).name}: {e}")
    print(f"wrote {n} pairs → {out_dir}")
    return n

print("Dataset builders ready.")
print("Instruments configured:", ", ".join(SOUNDFONTS))


## §13 — Dataset & dataloader

Random fixed-length crops. Mel is normalized to roughly zero-mean/unit-variance for stable regression; the constants are the standard log-mel range for music and are inverted before vocoding.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

MEL_MEAN, MEL_STD = -6.0, 2.5   # log-mel stats; invert before vocoding

class PairDataset(Dataset):
    def __init__(self, cache_dir, seq_frames=CFG["seq_frames"]):
        self.files = sorted(pathlib.Path(cache_dir).glob("pair_*.npz"))
        self.seq = seq_frames
        if not self.files:
            raise RuntimeError(f"no cached pairs in {cache_dir} — run §12 first")

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        d = np.load(self.files[i])
        roll, mel = d["roll"].astype(np.float32), d["mel"].astype(np.float32)
        T = min(roll.shape[2], mel.shape[1])
        s = random.randint(0, max(0, T - self.seq))
        roll, mel = roll[:, :, s:s+self.seq], mel[:, s:s+self.seq]
        # pad short tails
        if roll.shape[2] < self.seq:
            pad = self.seq - roll.shape[2]
            roll = np.pad(roll, ((0,0),(0,0),(0,pad)))
            mel  = np.pad(mel,  ((0,0),(0,pad)), constant_values=MEL_MEAN - 2*MEL_STD)
        mel = (mel - MEL_MEAN) / MEL_STD
        return torch.from_numpy(roll), torch.from_numpy(mel)

def make_loaders(cache_dir, val_frac=0.05):
    ds = PairDataset(cache_dir)
    n_val = max(1, int(len(ds) * val_frac))
    tr, va = torch.utils.data.random_split(
        ds, [len(ds)-n_val, n_val], generator=torch.Generator().manual_seed(CFG["seed"]))
    common = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], pin_memory=True)
    return (DataLoader(tr, shuffle=True, drop_last=True, **common),
            DataLoader(va, shuffle=False, **common))

print("Dataset ready.")


## §14 — The model

A frame-synchronous sequence model: `(3, 128, T)` conditioning → `(128, T)` mel.

1. **Input projection** — flatten `3 × 128 = 384` conditioning values per frame into `hidden`.
2. **Transformer encoder** — self-attention over time. This is what gives the model *context*:
   the sound of a note depends on what came before (pedal state, sympathetic resonance, whether
   the note is a repeat) and slightly on what follows.
3. **Output projection** — `hidden` → `n_mels`.

~15 M parameters at defaults. Small by modern standards, which is the point: it trains in hours, and
the task is genuinely narrow (one instrument, one job).

**Loss = L1 on log-mel.** L1 rather than L2 because it is less dominated by the loud
low-frequency bins and empirically gives sharper spectrograms.

In [ ]:
import torch, torch.nn as nn, math

class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=8192):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):            # (B, T, D)
        return x + self.pe[:, :x.size(1)]

class PianoRollToMel(nn.Module):
    """Conditioning piano roll → mel spectrogram. One instance per target instrument."""
    def __init__(self, cfg=CFG):
        super().__init__()
        d_in = cfg["n_cond_ch"] * cfg["n_pitches"]     # 3 × 128 = 384
        h    = cfg["hidden"]

        # Local temporal context before attention — cheap, and helps attacks.
        self.stem = nn.Sequential(
            nn.Conv1d(d_in, h, kernel_size=5, padding=2), nn.GELU(),
            nn.Conv1d(h,    h, kernel_size=5, padding=2), nn.GELU(),
        )
        self.pos = PositionalEncoding(h)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=h, nhead=cfg["n_heads"], dim_feedforward=h*4,
                dropout=cfg["dropout"], batch_first=True, norm_first=True, activation="gelu"),
            num_layers=cfg["n_layers"])
        self.head = nn.Sequential(
            nn.Conv1d(h, h, kernel_size=5, padding=2), nn.GELU(),
            nn.Conv1d(h, cfg["n_mels"], kernel_size=1),
        )

    def forward(self, roll):                 # (B, 3, 128, T)
        B, C, P, T = roll.shape
        x = roll.reshape(B, C * P, T)        # (B, 384, T)
        x = self.stem(x)                     # (B, H, T)
        x = self.pos(x.transpose(1, 2))      # (B, T, H)
        x = self.encoder(x)
        return self.head(x.transpose(1, 2))  # (B, n_mels, T)

def build_model(cfg=CFG, device="cuda"):
    m = PianoRollToMel(cfg).to(device)
    print(f"model: {sum(p.numel() for p in m.parameters())/1e6:.1f}M params")
    return m

def mel_loss(pred, target):
    """L1 on log-mel + a light emphasis on onset frames (large frame-to-frame deltas),
    because attacks matter perceptually far more than their pixel count suggests."""
    l1 = torch.nn.functional.l1_loss(pred, target)
    d_pred = pred[..., 1:] - pred[..., :-1]
    d_targ = target[..., 1:] - target[..., :-1]
    return l1 + 0.5 * torch.nn.functional.l1_loss(d_pred, d_targ)

print("Model defined.")


## §15 — Sanity overfit (run this FIRST)

Before any long run: can the model overfit a *single* example? If it cannot drive the loss to near
zero on one crop, something is structurally broken (shapes, normalization, alignment) and a long run
would only waste money discovering the same thing slowly.

Takes ~2 minutes. Loss should fall by well over an order of magnitude.

In [ ]:
def sanity_overfit(steps=300, device="cuda"):
    """Overfit one batch. Loss must collapse — otherwise the pipeline is broken."""
    train_loader, _ = make_loaders(CACHE_DIR)
    roll, mel = next(iter(train_loader))
    roll, mel = roll[:2].to(device), mel[:2].to(device)

    model = build_model(device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    model.train()
    first = None
    for i in range(steps):
        loss = mel_loss(model(roll), mel)
        opt.zero_grad(); loss.backward(); opt.step()
        if i == 0: first = loss.item()
        if i % 50 == 0 or i == steps-1:
            print(f"  step {i:4d}  loss {loss.item():.4f}")
    ratio = first / max(loss.item(), 1e-8)
    print(f"\nloss reduced {ratio:.0f}×")
    print("✅ pipeline sound" if ratio > 10 else
          "❌ NOT LEARNING — check shapes/normalization/alignment before the full run")
    return model

# model = sanity_overfit()
print("Uncomment to run once §12 has produced cached pairs.")


## §16 — Full training

Detached with `nohup` so an SSH drop or closed laptop does not kill it. Resumable.

**Expected:** ~6–12 h to a usable model on a 24 GB card; loss keeps improving slowly after. Watch
validation loss, but **trust your ears** — render audio samples periodically (the val hook does
this) and listen. Mel L1 correlates only loosely with perceived quality.

In [ ]:
training_script = r"""
import os, sys, math, random, pathlib, numpy as np, torch
from torch.utils.tensorboard import SummaryWriter
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from train_lib import CFG, make_loaders, build_model, mel_loss, CACHE_DIR, CKPT_DIR

def main():
    dev = "cuda"
    train_loader, val_loader = make_loaders(CACHE_DIR)
    model = build_model(device=dev)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-2, betas=(0.9, 0.99))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=CFG["lr"], total_steps=CFG["max_steps"],
        pct_start=CFG["warmup"]/CFG["max_steps"], anneal_strategy="cos")
    scaler = torch.cuda.amp.GradScaler()
    writer = SummaryWriter(str(CKPT_DIR / "tb"))

    # resume
    step = 0
    last = CKPT_DIR / "latest.pt"
    if last.exists():
        ck = torch.load(last, map_location=dev)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); step = ck["step"]
        print(f"resumed at step {step}", flush=True)

    model.train()
    best = float("inf")
    while step < CFG["max_steps"]:
        for roll, mel in train_loader:
            if step >= CFG["max_steps"]: break
            roll, mel = roll.to(dev, non_blocking=True), mel.to(dev, non_blocking=True)
            with torch.cuda.amp.autocast():
                loss = mel_loss(model(roll), mel)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1

            if step % 100 == 0:
                writer.add_scalar("train/loss", loss.item(), step)
                print(f"step {step}  loss {loss.item():.4f}  lr {sched.get_last_lr()[0]:.2e}", flush=True)

            if step % CFG["val_every"] == 0:
                model.eval(); vs = []
                with torch.no_grad():
                    for vr, vm in val_loader:
                        vs.append(mel_loss(model(vr.to(dev)), vm.to(dev)).item())
                v = float(np.mean(vs)); model.train()
                writer.add_scalar("val/loss", v, step)
                print(f"  VAL step {step}  loss {v:.4f}", flush=True)
                if v < best:
                    best = v
                    torch.save({"model": model.state_dict(), "cfg": CFG, "step": step, "val": v},
                               CKPT_DIR / "best.pt")
                    print(f"  ✓ new best {v:.4f}", flush=True)

            if step % CFG["save_every"] == 0:
                torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                            "sched": sched.state_dict(), "step": step}, CKPT_DIR / "latest.pt")

    torch.save({"model": model.state_dict(), "cfg": CFG, "step": step}, CKPT_DIR / "final.pt")
    print("TRAINING COMPLETE", flush=True)

if __name__ == "__main__":
    main()
"""
(ROOT / "train_poly.py").write_text(training_script)
print(f"wrote {ROOT/'train_poly.py'}")
print("\nNOTE: also export CFG/make_loaders/build_model/mel_loss to train_lib.py "
      "(or inline them) so the detached script can import them.")


In [ ]:
# Launch detached — survives SSH disconnects.
LOG = ROOT / f"train_{INSTRUMENT}.log"
print(f"""
cd {ROOT} && nohup python train_poly.py > {LOG} 2>&1 < /dev/null & disown

  tail -f {LOG}                       # follow
  tensorboard --logdir {CKPT_DIR/'tb'} --port 6006
""")

# %load_ext tensorboard
# %tensorboard --logdir {str(CKPT_DIR/"tb")} --port 6006


---

# Part VI · Inference — the complete pipeline

Everything assembled: polyphonic audio in, target instrument audio out.

In [ ]:
@torch.no_grad()
def convert(audio_path, target_instrument, model, vocoder,
            policy="ensemble", max_voices=None, device="cuda", out_path=None):
    """Polyphonic audio → same performance on `target_instrument`.

    1. transcribe   audio → note events        (pretrained Basic Pitch)
    2. voice        notes → target-playable    (rules, §6)
    3. encode       notes → piano roll         (§11)
    4. render       roll  → mel                (OUR TRAINED MODEL)
    5. vocode       mel   → waveform           (frozen BigVGAN)
    """
    import soundfile as sf

    # 1 — transcribe
    midi_data, _ = transcribe(audio_path, out_midi="_tmp.mid")
    n_notes = sum(len(i.notes) for i in midi_data.instruments)
    print(f"  1. transcribed  : {n_notes} notes")

    # 2 — voicing policy
    midi_data = apply_voicing(midi_data, target_instrument, policy=policy, max_voices=max_voices)
    n_after = sum(len(i.notes) for i in midi_data.instruments)
    print(f"  2. voiced       : {n_after} notes ({policy})"
          f"{f' — {n_notes-n_after} out of range' if n_after < n_notes else ''}")

    # 3 — piano roll
    roll = midi_to_pianoroll(midi_data)
    print(f"  3. piano roll   : {roll.shape}  ({roll.shape[2]/CFG['frame_rate']:.1f}s)")

    # 4 — render mel (chunked, with overlap, to bound memory on long inputs)
    model.eval()
    x = torch.from_numpy(roll).unsqueeze(0).to(device)
    CHUNK, OVER = 2048, 128
    mels = []
    for s in range(0, x.shape[-1], CHUNK - OVER):
        seg = x[..., s:s+CHUNK]
        if seg.shape[-1] < 16: break
        m = model(seg)
        mels.append(m[..., :-OVER] if s + CHUNK < x.shape[-1] else m)
    mel = torch.cat(mels, dim=-1)
    mel = mel * MEL_STD + MEL_MEAN            # de-normalize
    print(f"  4. mel          : {tuple(mel.shape)}")

    # 5 — vocode
    audio = mel_to_audio(mel.squeeze(0), vocoder)
    peak = audio.abs().max().item()
    if peak > 0: audio = audio / peak * 0.9
    print(f"  5. audio        : {len(audio)/CFG['sample_rate']:.1f}s  peak {peak:.3f}")

    out_path = out_path or f"converted_{target_instrument}.wav"
    sf.write(out_path, audio.numpy(), CFG["sample_rate"])
    print(f"  → {out_path}")
    return out_path

print("Inference pipeline ready.")


## §17 — Free baseline: per-note DDSP stacking

Before trusting the trained model, compare against this — it costs **nothing to train** because it
reuses ReTone's existing Engine-1 DDSP decoders.

Transcribe → for each note, synthesize a monophonic tone at that pitch through the existing DDSP
violin/flute/sax decoder → sum the voices.

**Known weaknesses** (worth hearing for yourself): summed independent voices share no room or
sympathetic resonance, so dense chords sound "stacked" rather than blended, and attacks are weaker
than a real instrument's. But it is a genuinely useful yardstick — if the trained model does not
clearly beat this, it is not ready.

In [ ]:
def ddsp_stack_baseline(midi_data, ddsp_engine, instrument="violin", sr=44100):
    """Render each transcribed note through the existing monophonic DDSP decoder, then sum.
    Requires ReTone's runpod/ddsp_engine.py (Engine 1)."""
    import numpy as np
    notes = [n for inst in midi_data.instruments if not inst.is_drum for n in inst.notes]
    if not notes: return np.zeros(sr, dtype=np.float32)

    total = int(max(n.end for n in notes) * sr) + sr
    mix = np.zeros(total, dtype=np.float32)

    for n in notes:
        dur = n.end - n.start
        if dur < 0.02: continue
        f0 = 440.0 * 2 ** ((n.pitch - 69) / 12)          # equal temperament
        t = np.arange(int(dur * sr)) / sr
        # A crude harmonic excitation; DDSP re-timbres it to the target instrument.
        exc = sum(np.sin(2*np.pi*f0*k*t) / k for k in range(1, 8)).astype(np.float32)
        exc *= np.minimum(1.0, t / 0.01) * np.exp(-t / max(dur, 0.1))   # attack + decay
        exc *= n.velocity / 127.0

        rendered, _ = ddsp_engine.render_mono(exc, sr, instrument)
        s = int(n.start * sr)
        e = min(s + len(rendered), total)
        mix[s:e] += rendered[:e-s]

    peak = np.abs(mix).max() + 1e-9
    return (mix / peak * 0.9).astype(np.float32)

print("DDSP-stacking baseline ready (needs Engine 1's ddsp_engine).")


## §18 — Evaluation

Mel L1 is what we optimize, but it is a weak proxy for "sounds good". Check these instead:

| Check | Question | How |
|---|---|---|
| **Note accuracy** | did the right notes come out? | re-transcribe the output, compare to input notes (F1) |
| **Timing** | is the groove intact? | onset-time RMS error, target < 20 ms |
| **Dynamics** | is expression preserved? | correlate frame RMS envelope, input vs output |
| **Timbre** | does it sound like the instrument? | **listen**; optionally FAD against real recordings |
| **Vocoder ceiling** | how much is us vs the vocoder? | §10 round-trip — you cannot beat it |

The re-transcription check is the strongest objective signal: if you transcribe the *output* and get
back the notes you put in, the performance survived the round trip.

In [ ]:
def evaluate_conversion(original_path, converted_path):
    """Objective checks on a conversion. Complements — does not replace — listening."""
    import librosa, numpy as np

    orig_midi, _ = transcribe(original_path,  out_midi="_eval_orig.mid")
    conv_midi, _ = transcribe(converted_path, out_midi="_eval_conv.mid")

    def note_set(md, tol=0.05):
        return {(n.pitch, round(n.start / tol)) for i in md.instruments for n in i.notes}

    a, b = note_set(orig_midi), note_set(conv_midi)
    tp = len(a & b)
    prec = tp / max(len(b), 1); rec = tp / max(len(a), 1)
    f1 = 2*prec*rec / max(prec+rec, 1e-9)
    print(f"  note F1        : {f1:.3f}   (precision {prec:.3f}, recall {rec:.3f})")

    yo, _ = librosa.load(original_path,  sr=CFG["sample_rate"], mono=True)
    yc, _ = librosa.load(converted_path, sr=CFG["sample_rate"], mono=True)
    n = min(len(yo), len(yc))
    ro = librosa.feature.rms(y=yo[:n], hop_length=CFG["hop_length"])[0]
    rc = librosa.feature.rms(y=yc[:n], hop_length=CFG["hop_length"])[0]
    m = min(len(ro), len(rc))
    corr = float(np.corrcoef(ro[:m], rc[:m])[0, 1])
    print(f"  dynamics corr  : {corr:.3f}   (>0.8 = expression preserved)")

    co = librosa.feature.spectral_centroid(y=yo[:n], sr=CFG["sample_rate"]).mean()
    cc = librosa.feature.spectral_centroid(y=yc[:n], sr=CFG["sample_rate"]).mean()
    print(f"  centroid       : {co:.0f} Hz → {cc:.0f} Hz   (should differ — timbre changed)")
    return {"note_f1": f1, "dynamics_corr": corr}

print("Evaluation ready.")


---

# Part VII · Deploy, cost, and honest expectations

## §19 — Drop into the ReTone worker

The trained checkpoint is a plain `state_dict`:

```bash
mkdir -p runpod/poly_models/strings
scp -P <port> root@<pod>:/workspace/retone_poly/ckpt/strings/best.pt runpod/poly_models/strings/model.pt
```

Then, mirroring `runpod/ddsp_engine.py` and `runpod/after_engine.py`:

1. **`runpod/poly_engine.py`** — a `PolyEngine` class with
   `render(audio, sr, target_instrument) -> (audio, sr)`, wrapping the §16 `convert()` pipeline.
2. **`runpod/handler.py`** — add `_get_poly()` (lazy, like the others — a module-level import blocks
   `runpod.serverless.start()`), and a `"poly"` branch in `_do_tone_transfer`'s engine dispatch.
3. **`backend/app/services/tone_transfer.py`** — populate `POLY_INSTRUMENTS`.
4. **`frontend/.../NoteEditor.tsx`** — a `"poly:<instrument>"` `<optgroup>`.
5. **`.gitignore`** — whitelist the checkpoint if under 100 MB, else fetch at Dockerfile build time
   via `runpod/download_models.py`.

The engine-dispatch plumbing already exists (`engine` field on the job payload), so this is
additive.

## §20 — Cost

| Stage | Time | Cost @ ~$0.75/h (RTX 4090) |
|---|---|---|
| Data prep (render + cache, CPU) | 2–6 h | ~$0 (CPU) |
| Sanity overfit | 2 min | ~$0.03 |
| Training to usable | 6–12 h | **$5–10** |
| Training to converged | 24–36 h | $18–27 |

**Per instrument: roughly $5–10 for a first usable model.** Five instruments ≈ $25–50. That is
comparable to the DDSP training runs, and far cheaper than the latent-diffusion alternatives —
because the model is small and the task is narrow.

Do §15 (2 min, ~$0.03) before every long run.

## §21 — What to expect, honestly

**Should be good:** note accuracy and timing (they are handed to the model as data, not inferred);
per-instrument timbre identity, for the same reason DDSP's is good — the model only ever learns one
instrument.

**Will be harder:** the "air" — room, sympathetic resonance, mic character. Mel + vocoder captures
much of this but not all. Expressive nuance that MIDI cannot represent (bow pressure, exact
articulation) is lost at Stage 1 and must be re-invented.

**The honest ceiling:** this pipeline can only be as good as (a) the transcription and (b) the
vocoder round-trip. Measure both (§5, §10) before blaming the model.

**Versus monophonic DDSP:** likely close but not identical. DDSP works directly on the source's
*continuous* f0 and loudness curves — every micro-fluctuation of vibrato and dynamics survives.
Quantizing to discrete note events necessarily discards some of that. Expect very good musical
accuracy, with slightly less of the "alive" quality on solo lines. For polyphony, where DDSP cannot
compete at all, this should be a clear win.

## §22 — Kill criteria

Stop and reconsider if:

1. **§5 transcription is poor on your material** — everything downstream inherits it. Fix Stage 1 or
   change approach.
2. **§10 vocoder round-trip sounds bad** — wrong mel parameters (most likely), or the vocoder is a
   poor fit for this material.
3. **§15 sanity overfit does not converge** — structural bug; do not start a long run.
4. **Validation plateaus early with bad audio** — likely too little data diversity; add augmentation
   and more varied MIDI before adding parameters.

---

# References

**Transcription (Stage 1)**
- [Basic Pitch](https://github.com/spotify/basic-pitch) · [paper](https://arxiv.org/abs/2203.09893) — lightweight polyphonic, Apache-2.0
- [MT3](https://github.com/magenta/mt3) · [paper](https://arxiv.org/abs/2111.03017) — multi-instrument transformer
- [Onsets and Frames](https://arxiv.org/abs/1710.11153) — the piano-transcription lineage

**Synthesis (Stage 2)**
- [DDSP](https://github.com/magenta/ddsp) · [paper](https://openreview.net/pdf?id=B1x1ma4tDr) — our Engine 1; Google's Tone Transfer
- [DDSP-Piano](https://github.com/lrenault/ddsp-piano) · [paper](https://hal.science/hal-04073770) — polyphonic DDSP for piano
- [MIDI-DDSP](https://github.com/magenta/midi-ddsp) · [paper](https://arxiv.org/abs/2112.09312) — hierarchical note→expression→audio
- [Spectrogram Diffusion](https://arxiv.org/abs/2206.05408) — MIDI→spectrogram, multi-instrument, Slakh-trained

**Vocoders**
- [BigVGAN](https://github.com/NVIDIA/BigVGAN) · [paper](https://arxiv.org/abs/2206.04658) — what we use, MIT
- [HiFi-GAN](https://arxiv.org/abs/2010.05646) · [Vocos](https://github.com/gemelo-ai/vocos) · [DAC](https://github.com/descriptinc/descript-audio-codec)

**Audio-to-audio (the alternative we rejected)**
- [AFTER](https://github.com/acids-ircam/AFTER) · [paper](https://arxiv.org/abs/2408.00196) — tested; identity not in weights
- [RAVE](https://github.com/acids-ircam/RAVE) · [WaveTransfer](https://arxiv.org/abs/2409.15321) · [DiffTransfer](https://github.com/lucacoma/DiffTransfer)

**Datasets**
- [MAESTRO](https://magenta.withgoogle.com/datasets/maestro) ⚠️NC · [Slakh2100](https://zenodo.org/records/4599666) ·
  [GuitarSet](https://zenodo.org/records/3371780) · [URMP](https://labsites.rochester.edu/air/datasets/URMP.html) ·
  [CocoChorales](https://magenta.tensorflow.org/datasets/cocochorales) · [E-GMD](https://magenta.withgoogle.com/datasets/e-gmd) ·
  [Lakh MIDI](https://colinraffel.com/projects/lmd/)

**Sample libraries**
- [Salamander Grand Piano](https://freepats.zenvoid.org/Piano/SalamanderGrandPiano/) ·
  [Virtual Playing Orchestra](http://virtualplayingorchestra.com/) ·
  [GeneralUser GS](https://www.schristiancollins.com/generaluser.php)

**Sim-to-real**
- [Zehren et al. 2024](https://arxiv.org/html/2407.19823) — synthetic→real gap; preset diversity beats raw hours
- [Score-informed separation](https://arxiv.org/html/2503.07352v1) — +3.19 dB SDR from avoiding synthetic-audio dependence

**Related ReTone notebooks**
- `train_ddsp_48k.ipynb` — Engine 1, monophonic DDSP (validated end-to-end)
- `train_after_orchestral.ipynb` — Engine 2 v1, latent diffusion (superseded by this)